In [1]:
import sqlite3
import pandas as pd
import os
import csv

In [ ]:
DATABASE = "....../v33_koondkorpus_transaktsioonid_v04_2.db"

LINE_DATA_TABLE = "lines_class_info4"

FILTERED_CLASS_TABLE = "lines_class_info4_n80"

LARGE_DATA_FILE = "....../n80_examples_large_v2.csv"

LARGE_DATA_FILE_SORTED = "...../n80_examples_large_v2_sorted.csv"


## see teeb korrektse uue andmefaili v2

### andmetabelid

In [3]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

### graafiku punktide info

In [4]:
query = f"""SELECT verb, verb_compound, morph_case, log2_ratio, level,unique_lemmas, ann_unique_lemmas, 
            not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated, verb_case_count
            FROM {LINE_DATA_TABLE}
            """

class_info = pd.read_sql(query, conn)
class_info

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,-9.965784,-,1,NaN,5.0,-,0,1,1,7,8
1,abistama,,all,-9.965784,-,1,NaN,12.0,-,0,1,1,15,16
2,aeglustama,,all,-9.965784,-,1,NaN,7.0,-,0,1,1,8,9
3,aerutama,,in,-9.965784,-,1,NaN,5.0,-,0,3,3,10,13
4,aevastama,,in,9.965784,-,1,1.0,6.0,-,1,0,1,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,-,1313,331.0,2522.0,-,1022,4009,5031,17243,22274
20936,õppima,,in,5.233872,n90,530,463.0,985.0,0.0,5720,152,5872,5891,11763
20937,ütlema,,ad,-5.370614,n10,415,104.0,1162.0,0.0,295,12205,12500,9966,22466
20938,ütlema,,all,0.271387,n70,1131,189.0,2337.0,1.0,9354,7750,17104,40742,57846


### võtta ainult n80 tsooni lõksud

In [5]:
filtered_class = class_info[class_info["level"]=="n80"]
filtered_class = filtered_class.sort_values(["olulisus"])

In [6]:
filtered_class['olulisus'] = filtered_class['olulisus'].astype(float)

In [7]:
filtered_class

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
19889,peatuma,,in,3.551257,n80,282,257.0,337.0,0.00000,973,83,1056,858,1914
19994,möllama,,in,3.921070,n80,196,177.0,261.0,0.00000,409,27,436,495,931
20637,leiduma,,in,3.086569,n80,518,415.0,1689.0,0.00000,1614,190,1804,4760,6564
20673,naasma,,el,3.210249,n80,287,236.0,222.0,0.00000,907,98,1005,439,1444
19859,süttima,,in,3.879146,n80,199,177.0,233.0,0.00000,515,35,550,661,1211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18469,nappima,,in,3.496426,n80,140,114.0,198.0,0.00006,316,28,344,382,726
18887,sõitma,edasi,el,5.741467,n80,55,53.0,26.0,0.00007,107,2,109,30,139
20015,teatama,,ill,5.235216,n80,54,51.0,82.0,0.00008,113,3,116,640,756
20194,ootama,,ill,3.882643,n80,108,93.0,132.0,0.00008,236,16,252,231,483


In [8]:
filtered_class.to_sql(FILTERED_CLASS_TABLE, conn, if_exists="replace", index=False)

335

### võtta spatial_obl tabelist näitelaused koos vajaliku infoga

In [27]:
# spatial obl_tabelist lõksud, mis on lines_class_info4_n80 tabelis
# iga lõksu kohta max 500 lemmat ja iga unikaalse lemma kohta 1 näide


query = """
WITH cleaned AS (
    -- Step 1 & 2: match subset table + remove rows with timex_tag NOT NULL
    SELECT
        d.head_id,
        d.form,
        d.lemma,
        d.verb,
        d.verb_compound,
        d.morph_case,
        d.sentence,
        d.sentence_id,
        d.timex_tag,
        d.ekilex_tag,
        d.ner_tag
    FROM spatial_obl AS d
    JOIN lines_class_info4_n80 AS s
      ON d.verb = s.verb
     AND d.verb_compound = s.verb_compound
     AND d.morph_case = s.morph_case
    WHERE d.timex_tag IS NULL
),

distinct_lemmas AS (
    -- Step 3 & 4: for each lemma, pick ONE sentence deterministically
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case, lemma
            ORDER BY sentence_id   -- choose best or earliest sentence
        ) AS rn_per_lemma
    FROM cleaned
),

limited AS (
    -- Step 5: limit to 500 unique lemmas per (verb, verb_compound, morph_case)
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case
            ORDER BY lemma        -- choose 500 lexicographically smallest lemmas
        ) AS rn_group_limit
    FROM distinct_lemmas
    WHERE rn_per_lemma = 1      -- keep only one sentence per lemma
)

-- Step 6: final output
SELECT
    head_id,
    form,
    lemma,
    verb,
    verb_compound,
    morph_case,
    sentence,
    sentence_id,
    timex_tag,
    ekilex_tag,
    ner_tag
FROM limited
WHERE rn_group_limit <= 500
ORDER BY verb, verb_compound, morph_case, lemma;

"""


spatial_obl_ex = pd.read_sql(query, conn)


In [28]:
spatial_obl_ex

,head_id,form,lemma,verb,verb_compound,morph_case,sentence,sentence_id,timex_tag,ekilex_tag,ner_tag
0,23747396,17das,17.,ajama,,in,"Mingi aeg ajasin täna 17das servus kõiki Great Forge'i hüppama , isegi sain mõned",15208349,None,None,None
1,18645542,Aafrikas,Aafrika,ajama,,in,"Kas teadsite , et Brechti suu oli nagu mustade hauakividega surnuaed , et Hemingway ajas Aafrikas pea kiilaks ja lõhkus neegritüdrukuga välivoodi , et Bertrand Russell polnud võimeline endale teed keetma isegi siis , kui tal oli käes kirjalik juhend , kuidas seda teha .",11648113,None,location,LOC
2,26620253,Afganistanis,Afganistan,ajama,,in,"Afganistanis ja Iraagis musklid suureks ajanud Bush on olnud valmis kohe-kohe karistama « kurjuse telje » põhiliiget Iraani , ent mida kehvemalt on käinud käbarad kahel sõjatandril , seda vähemmõjuvaks on muutunud ka Bushi jutt Iraanist , mida nüüd , ennäe , väisas USAs ja Euroopas igast uksest ja aknast sisse lastud sõber Putin ise .",17346120,None,location,LOC
3,25724171,Alžeerias,Alžeeria,ajama,,in,"Alžeerias ajas mind nii naerma - ümberringi ainult liiv , liiv , liiv ...",16719535,None,location,LOC
4,17515699,Ameerikas,Ameerika,ajama,,in,"Lõppude lõpuks , kui Ameerikas ajavad metalpopi liini Smashing Pumpkins ja Foo Fighters , siis oleme meie siin Blindi väärt küll .",10936172,None,location,LOC
...,...,...,...,...,...,...,...,...,...,...,...
75407,21960802,väikelinnas,väikelinn,üürima,,in,Arnold üürib väikelinnas korterit .,13774030,None,location,None
75408,6229577,võõrastemajas,võõrastemaja,üürima,,in,"Haritud keskealine mees meenutab üht oma kunagist armulugu , mis algab sellega , et ta reisib välismaale ja üürib võõrastemajas toa .",3867933,None,location,None
75409,9515834,äärelinnas,äärelinn,üürima,,in,Noormees üürib väikese korteri äärelinnas .,5927429,None,location,None
75410,2889349,ühiselamus,ühiselamu,üürima,,in,"Üliõpilane Viljar ( 22 ) üürib tuba Mustamäel Vilde teel poollagunenud ühiselamus , sest peab kalli korteri üürimist ebaotstarbekaks ning isikliku elamispinna ostuks pole raha .",1815434,None,location,None


In [30]:
# shuffle
df = spatial_obl_ex.sample(frac=1)

In [31]:
df.to_csv(LARGE_DATA_FILE, encoding="utf-8", index = False,sep=",", quoting=csv.QUOTE_MINIMAL)

In [32]:
spatial_obl_ex.to_csv(LARGE_DATA_FILE_SORTED, encoding="utf-8", index = False,sep=",", quoting=csv.QUOTE_MINIMAL)

In [33]:
conn.close()

In [2]:
df2 = pd.read_csv(LARGE_DATA_FILE, encoding="utf-8", sep=",")

In [3]:
counts2 = df2.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,verb,verb_compound,morph_case,count
332,õpetama,NaN,in,500
0,ajama,NaN,in,500
3,andma,NaN,ill,500
274,tegelema,NaN,in,500
285,tooma,NaN,adit,500
...,...,...,...,...
272,tarnima,NaN,ill,37
296,turustama,NaN,in,35
121,lendama,edasi,ill,35
241,soetama,NaN,ill,34
